In [62]:
import h5py
import numpy as np
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import torch
from torch_geometric.data import Data


def quake_h5_to_pyg(path, n_neighbors=15, n_pcs=50):
    """
    Read Quake_Smart-seq2_Trachea.h5 and convert it to a PyTorch Geometric graph.

    Parameters
    ----------
    path : str
        Path to the HDF5 file.
    n_neighbors : int
        Number of neighbors used to build the KNN graph.
    n_pcs : int
        Number of principal components used for the graph.

    Returns
    -------
    data : torch_geometric.data.Data
        PyG graph object
    adata : AnnData
        AnnData object containing the processed dataset
    """

    # --- read file ---
    with h5py.File(path, "r") as f:

        grp = f["exprs"]

        X = sp.csr_matrix(
            (grp["data"][:], grp["indices"][:], grp["indptr"][:]),
            shape=grp["shape"][:]
        )

        cells = f["obs_names"][:].astype(str)
        genes = f["var_names"][:].astype(str)

        y = f["obs"]["cluster"][:]

    # --- build AnnData ---
    adata = ad.AnnData(X)
    adata.obs_names = cells
    adata.var_names = genes
    adata.obs["cluster"] = y

    # --- Scanpy pipeline ---
    sc.pp.pca(adata, n_comps=n_pcs)
    sc.pp.neighbors(adata, n_neighbors=n_neighbors)

    # adjacency matrix
    A = adata.obsp["connectivities"]

    edge_index = np.vstack(A.nonzero())
    edge_index = torch.tensor(edge_index, dtype=torch.long)

    x = torch.tensor(adata.obsm["X_pca"], dtype=torch.float)
    y = torch.tensor(adata.obs["cluster"].values)

    data = Data(x=x, edge_index=edge_index, y=y)

    print("Cells:", data.num_nodes)
    print("Edges:", data.num_edges)
    print("Clusters:", len(np.unique(y.numpy())))

    return data, adata

In [64]:
data, adata= quake_h5_to_pyg("10X_PMBC/10X_PBMC.h5")

KeyError: "Unable to open object (object 'exprs' doesn't exist)"

In [73]:
import h5py

with h5py.File("10X_PBMC.h5", "r") as f:
    def show(name, obj):
        print(name)
    f.visititems(show)

AttributeError: 'Dataset' object has no attribute 'items'

In [ ]:
import h5py

with h5py.File("10X_PBMC.h5", "r") as f:
    def show(name, obj):
        print(name)
    f.visititems(show)

In [20]:
import h5py

def read_xy_h5(path="data/paul15.h5"):
    with h5py.File(path, "r") as f:
        X = f["X"][:]   # expression matrix
        y = f["Y"][:]   # labels

    print("X shape:", X.shape)
    print("Number of cells:", X.shape[0])

    return X, y

In [21]:
read_xy_h5()

FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = 'data/paul15.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [6]:
import GEOparse

# Download GEO dataset
gse = GEOparse.get_GEO("GSE12345", destdir="HArc-ME")

# Access metadata
print(gse.metadata)

# Access samples
print(gse.gsms.keys())

# Expression table example
for gsm_name, gsm in gse.gsms.items():
    print(gsm.table.head())

05-Mar-2026 22:39:27 DEBUG utils - Directory HArc-ME already exists. Skipping.
05-Mar-2026 22:39:27 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE12nnn/GSE12345/soft/GSE12345_family.soft.gz to HArc-ME/GSE12345_family.soft.gz
100%|██████████| 15.8M/15.8M [00:01<00:00, 10.5MB/s]
05-Mar-2026 22:39:29 DEBUG downloader - Size validation passed
05-Mar-2026 22:39:29 DEBUG downloader - Moving /var/folders/46/r35cp70x5l7bsj60qqttzmjc0000gn/T/tmpwts3r3cn to /Users/be-cool/Documents/DOCS PROFESSIONELS/Research/CODES/PROJETS/scGMCM-VGAE/data/HArc-ME/GSE12345_family.soft.gz
05-Mar-2026 22:39:29 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE12nnn/GSE12345/soft/GSE12345_family.soft.gz
05-Mar-2026 22:39:29 INFO GEOparse - Parsing HArc-ME/GSE12345_family.soft.gz: 
05-Mar-2026 22:39:29 DEBUG GEOparse - DATABASE: GeoMiame
05-Mar-2026 22:39:29 DEBUG GEOparse - SERIES: GSE12345
05-Mar-2026 22:39:29 DEBUG GEOparse - PLATFORM: GPL570
/Users/be-cool

{'title': ['Global gene expression profiling of human pleural mesotheliomas'], 'geo_accession': ['GSE12345'], 'status': ['Public on Jul 30 2009'], 'submission_date': ['Aug 05 2008'], 'last_update_date': ['Mar 25 2019'], 'pubmed_id': ['19753302'], 'summary': ['The goal of our study was to molecularly dissect mesothelioma tumor pathways by mean of microarray technologies in order to identify new tumor biomarkers, that could be used as early diagnostic markers and possibly as specific molecular therapeutic targets. We performed Affymetrix U133A plus 2.0 microarray analysis comparing 9 human pleural mesotheliomas with 4 normal pleural specimen. Stringent statistical feature selection detected a set of differentially expressed genes that were further evaluated to identify potential biomarkers to be used in early diagnostics. Selected genes were confirmed by RT-PCR. As reported by other mesothelioma profiling studies, most of genes are involved in G2/M transition. Our list contains several g

In [18]:
import scanpy as sc

adata = sc.read_10x_mtx(
    "HArc-ME/",
    var_names="gene_symbols",
    cache=True
)

print("Cells:", adata.n_obs)
print("Genes:", adata.n_vars)

FileNotFoundError: Did not find file HArc-ME/matrix.mtx.gz.

In [2]:
import pandas as pd

data=pd.read_csv("HArc-ME/GSE93374_cell_metadata.txt", sep="\t")
print(data.info)

<bound method DataFrame.info of                           1.ID 2.group 3.batches 4.sex 5.Diet 6.FvF  \
0            arc1_AAAATCTCTCCC    arc1        b1     M   Chow   Fed   
1            arc1_AAAGCGGATGCG    arc1        b1     M   Chow   Fed   
2            arc1_AACGTGTCTAAC    arc1        b1     M   Chow   Fed   
3            arc1_ACTATACTCTCA    arc1        b1     M   Chow   Fed   
4            arc1_AGCGGCCGACCA    arc1        b1     M   Chow   Fed   
...                        ...     ...       ...   ...    ...   ...   
21081   FemaleFed_GCTTTCACAGTN    FFed        b6     F   Chow   Fed   
21082  MaleFasted_AGGCTAGTGAGN   MFast        b6     M   Fast  Fast   
21083  MaleFasted_GACTATTTAGAN   MFast        b6     M   Fast  Fast   
21084  MaleFasted_TTTAGCGACGGC   MFast        b6     M   Fast  Fast   
21085     MaleFed_TTTGGTTACGAN    MFed        b6     M   Chow   Fed   

      7.clust_all 8.clust_all_neurons 9.clust_all_micro 10.clust_neurons  \
0             a03                 a03  

In [11]:
import numpy as np
for col in data.columns:
    b=np.unique(data[col].values)
    print(f"{col}:{len(b)} ,{b}")

1.ID:21086 ,['Chow_AAAAAAGATACT' 'Chow_AAAACGAGTACC' 'Chow_AAAACTAAAACA' ...
 'arc3_TTTTGAGGGTGC' 'arc3_TTTTGTGTATTA' 'arc3_TTTTTGCCCCTA']
2.group:11 ,['Ch10' 'FFast' 'FFed' 'HFD' 'MFast' 'MFed' 'Refed' 'UFast' 'arc1' 'arc2'
 'arc3']
3.batches:6 ,['b1' 'b2' 'b3' 'b4' 'b5' 'b6']
4.sex:3 ,['F' 'M' 'U']
5.Diet:5 ,['Ch10' 'Chow' 'Fast' 'HFD' 'Refed']
6.FvF:5 ,['Ch10' 'Fast' 'Fed' 'HFD' 'Refed']
7.clust_all:21 ,['a01' 'a02' 'a03' 'a04' 'a05' 'a06' 'a07' 'a08' 'a09' 'a10' 'a11' 'a12'
 'a13' 'a14' 'a15' 'a16' 'a17' 'a18' 'a19' 'a20' 'miss']
8.clust_all_neurons:16 ,['a01' 'a02' 'a03' 'a04' 'a05' 'a06' 'a07' 'a08' 'a09' 'a10' 'a11' 'a12'
 'a19' 'a20' 'miss' 'neuron']
9.clust_all_micro:37 ,['miss' 's01' 's02' 's03' 's04' 's05' 's06' 's07' 's08' 's09' 's10' 's11'
 's12' 's13' 's14' 's15' 's16' 's17' 's18' 's19' 's20' 's21' 's22' 's23'
 's24' 's25' 's26' 's27' 's28' 's29' 's30' 's31' 's32' 's33' 's34' 's35'
 's36']
10.clust_neurons:35 ,['miss' 'n01' 'n02' 'n03' 'n04' 'n05' 'n06' 'n07' 'n08' 'n09' 

TypeError: '<' not supported between instances of 'float' and 'str'

In [6]:
print(data.columns)

Index(['1.ID', '2.group', '3.batches', '4.sex', '5.Diet', '6.FvF',
       '7.clust_all', '8.clust_all_neurons', '9.clust_all_micro',
       '10.clust_neurons', '11.Sex_pred', 'Unnamed: 11', 'All Cell Clusters',
       'All Cell Subclusters', 'Neuron Subclusters'],
      dtype='object')


In [ ]:
import h5py
import anndata as ad
path="data/Quake_Smart-seq2_Lung/Quake_Smart-seq2_Lung.h5"
with h5py.File(path, "r") as f:
        X = f["X"][:]
        y = f["Y"][:]

    # build AnnData
adata = ad.AnnData(X)
adata.obs["label"] = y

In [ ]:
def evaluate(y_true, y_pred):
    acc= cluster_acc(y_true, y_pred)
    f1=0
    nmi = normalized_mutual_info_score(y_true, y_pred)
    ari = adjusted_rand_score(y_true, y_pred)
    homo = homogeneity_score(y_true, y_pred)
    comp = completeness_score(y_true, y_pred)
    return acc, f1, nmi, ari, homo, comp


In [1]:
import pandas as pd
import numpy as np


def load_campell(data_path):
    label_data=pd.read_csv(f"ç/GSE93374_cell_metadata.txt", sep="\t")
    data=pd.read_csv(f"{data_path}/GSE93374_Merged_all_020816_DGE.txt", sep="\t")

    label_name=[]
    labels=[]
    for i,col  in enumerate(data.columns):
        label_name.append(label_data.loc[label_data["1.ID"] == col,"1.ID"].iloc[0])
        labels.append(label_data.loc[label_data["1.ID"] == col,"2.group"].iloc[0])

     if set(data.columns)== set(label_name):
        for i,col  in enumerate(data.columns):
            if col != label_name[i]:
                raise ValueError("Mismatch in cell ID")
     elif set(data.columns) != set(label_name):
         raise ValueError("Mismatch in cell ID")

    X = data.T
    return X, labels




               arc1_TACTAACAGTAN  arc1_CCGCGAGCTCTT  arc1_GTTGCACGGATA  \
0610005C13Rik                  0                  0                  0   
0610007P14Rik                  1                  0                  0   
0610009B22Rik                  6                  5                  3   
0610009E02Rik                  0                  0                  1   
0610009L18Rik                  0                  2                  0   

               arc1_CTGGCATTTTAT  arc1_TGCAACGACTAT  arc1_CCGTAATACTTN  \
0610005C13Rik                  0                  0                  1   
0610007P14Rik                  0                  0                  0   
0610009B22Rik                  2                  1                  1   
0610009E02Rik                  0                  0                  0   
0610009L18Rik                  0                  0                  0   

               arc1_CAATCCGCTGGN  arc1_ACAAGTCATGAT  arc1_ACGAGCCCTCCA  \
0610005C13Rik                  0    

In [1]:
import pandas as pd

data=pd.read_csv("Baron/GSM2230757_human1_umifm_counts.csv")
print(data.info)

<bound method DataFrame.info of                        Unnamed: 0               barcode    assigned_cluster  \
0     human1_lib1.final_cell_0001   GATGACGGAC-GGTGGGAT              acinar   
1     human1_lib1.final_cell_0002   GAGCGTTGCT-ACCTTCTT              acinar   
2     human1_lib1.final_cell_0003     CTTACGGG-CCATTACT              acinar   
3     human1_lib1.final_cell_0004   GATGTACACG-TTAAACTG              acinar   
4     human1_lib1.final_cell_0005   GAGATTGCGA-GTCGTCGT              acinar   
...                           ...                   ...                 ...   
1932  human1_lib3.final_cell_0736   GAGAGAGTAT-GATTTACC         endothelial   
1933  human1_lib3.final_cell_0737  TGATTCGCTGG-CTTCTGGA                beta   
1934  human1_lib3.final_cell_0738     GCTTACCT-GGCATGCT         endothelial   
1935  human1_lib3.final_cell_0739     CGGCACAT-TGGCCTGT                beta   
1936  human1_lib3.final_cell_0740     TGCCTCAC-ACATCTAT  quiescent_stellate   

      A1BG  A1CF  A

True

arc1_TACTAACAGTAN and  arc1_TACTAACAGTAN are the same | Class: arc1
arc1_CCGCGAGCTCTT and  arc1_CCGCGAGCTCTT are the same | Class: arc1
arc1_GTTGCACGGATA and  arc1_GTTGCACGGATA are the same | Class: arc1
arc1_CTGGCATTTTAT and  arc1_CTGGCATTTTAT are the same | Class: arc1
arc1_TGCAACGACTAT and  arc1_TGCAACGACTAT are the same | Class: arc1
arc1_CCGTAATACTTN and  arc1_CCGTAATACTTN are the same | Class: arc1
arc1_CAATCCGCTGGN and  arc1_CAATCCGCTGGN are the same | Class: arc1
arc1_ACAAGTCATGAT and  arc1_ACAAGTCATGAT are the same | Class: arc1
arc1_ACGAGCCCTCCA and  arc1_ACGAGCCCTCCA are the same | Class: arc1
arc1_GAATTAGGGGTC and  arc1_GAATTAGGGGTC are the same | Class: arc1
arc1_CCCTCCTTAGAT and  arc1_CCCTCCTTAGAT are the same | Class: arc1
arc1_GTCACCGGAATT and  arc1_GTCACCGGAATT are the same | Class: arc1
arc1_CTAGATGATTTG and  arc1_CTAGATGATTTG are the same | Class: arc1
arc1_AAGGTAACTGTN and  arc1_AAGGTAACTGTN are the same | Class: arc1
arc1_AGTCTCCCAAGC and  arc1_AGTCTCCCAAGC are the